# T02 Pose Estimation — chạy trên Google Colab (GPU)

Notebook này: **upload code T01/T02 (nhỏ) lên Drive**, dùng chung với **dataset `MuseumVisitors.zip` đã upload sẵn**, rồi chạy `estimate_pose.py` bằng GPU của Colab (nhanh hơn M1 CPU/MPS).

## Bước 0 — chuẩn bị file zip code TRÊN MÁY LOCAL (chạy trong Terminal, không phải trong notebook)

```bash
cd /Users/nguyenvokhang/Downloads/ExhibitFlow
zip -r exhibitflow_code.zip \
    Museum/pose_rule_anomaly/src \
    Museum/pose_rule_anomaly/outputs/manifest.csv \
    Museum/pose_rule_anomaly/outputs/annotations.jsonl \
    -x "*/__pycache__/*"
```

File này chỉ ~750KB (không chứa ảnh) nên upload lên Google Drive rất nhanh. **Mỗi lần code thay đổi, hãy upload lại file này và ghi đè `exhibitflow_code.zip` cũ trên Drive.** Dataset (`MuseumVisitors.zip`) không cần upload lại — dùng đúng file bạn đã upload trước đó, dù cấu trúc thư mục bên trong nó thế nào, notebook sẽ tự tìm. Script sẽ tự ánh xạ các đường dẫn ảnh tuyệt đối được tạo trên Mac sang thư mục dataset hiện tại của Colab và dừng ngay nếu còn thiếu ảnh.

## Lưu ý quan trọng về Colab free tier
- Phiên free có thể bị ngắt nếu idle ~90 phút, và giới hạn cứng ~12 giờ.
- Script `estimate_pose.py` **không có khả năng resume** — nếu phiên bị ngắt giữa chừng, lần chạy lại sẽ ghi đè và bắt đầu lại từ đầu (frame 0). Hãy để tab Colab mở và tương tác (chạy cell, xem log) để tránh bị ngắt do idle.
- Đã sửa `pick_device()` trong script để tự ưu tiên CUDA (GPU Colab) > MPS (Mac) > CPU, nên chạy trên Colab sẽ dùng đúng GPU T4 thay vì rơi về CPU.

In [ ]:
# Bước 1 — kiểm tra GPU runtime (Runtime > Change runtime type > GPU phải được chọn trước khi chạy cell này)
!nvidia-smi

In [ ]:
# Bước 2 — mount Google Drive (nơi bạn đã upload exhibitflow_payload.zip)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Bước 3 — chỉnh 2 đường dẫn này đúng tới chỗ bạn upload trong Drive
DATASET_ZIP = "/content/drive/MyDrive/MuseumVisitors.zip"      # <-- file dataset đã upload từ trước
CODE_ZIP     = "/content/drive/MyDrive/exhibitflow_code.zip"   # <-- file code nhỏ mới upload

import os
for p in (DATASET_ZIP, CODE_ZIP):
    assert os.path.exists(p), f"Khong tim thay {p} — kiem tra lai duong dan trong Drive"
    print("OK, found:", p, f"({os.path.getsize(p) / 1e6:.1f} MB)")

In [ ]:
# Bước 4 — giải nén cả 2 file zip, ghép đúng cấu trúc thư mục
PROJECT_ROOT = "/content/ExhibitFlow"
EXPECTED_DATASET = f"{PROJECT_ROOT}/Museum/dataset/MuseumVisitors"
DATASET_TMP = "/content/_dataset_tmp"

!mkdir -p {PROJECT_ROOT}/Museum/dataset {DATASET_TMP}

# 4a. code zip: cấu trúc bên trong đã đúng Museum/pose_rule_anomaly/... nên giải nén thẳng vào PROJECT_ROOT
!unzip -q -o "{CODE_ZIP}" -d {PROJECT_ROOT}

# 4b. dataset zip: không biết chắc cấu trúc nội bộ, nên giải nén ra thư mục tạm rồi tự dò
!unzip -q -o "{DATASET_ZIP}" -d {DATASET_TMP}

import subprocess, shutil, os

# tìm file "annotations.dat" để xác định đúng gốc dataset, bất kể nó nằm sâu bao nhiêu lớp thư mục
found = subprocess.run(
    ['bash', '-c', f'find {DATASET_TMP} -name annotations.dat'],
    capture_output=True, text=True,
).stdout.strip().splitlines()

assert found, f"Khong tim thay annotations.dat trong {DATASET_TMP} — kiem tra lai noi dung MuseumVisitors.zip"
dataset_root = os.path.dirname(found[0])
print("Tim thay dataset root:", dataset_root)

if os.path.exists(EXPECTED_DATASET):
    shutil.rmtree(EXPECTED_DATASET)
shutil.move(dataset_root, EXPECTED_DATASET)
shutil.rmtree(DATASET_TMP, ignore_errors=True)

!echo '--- top level ---' && ls {PROJECT_ROOT}/Museum
!echo '--- dataset ---' && ls {EXPECTED_DATASET}
!echo '--- pose_rule_anomaly ---' && ls {PROJECT_ROOT}/Museum/pose_rule_anomaly

In [ ]:
# Bước 5 — sanity check: số dòng manifest/annotations và số ảnh mỗi camera phải khớp với local
REPO = f"{PROJECT_ROOT}/Museum/pose_rule_anomaly"
DATASET = f"{PROJECT_ROOT}/Museum/dataset/MuseumVisitors"

!wc -l {REPO}/outputs/manifest.csv {REPO}/outputs/annotations.jsonl
for cam in (1, 2, 3):
    import subprocess
    n = subprocess.run(['bash', '-c', f'ls {DATASET}/frames/{cam} | wc -l'], capture_output=True, text=True).stdout.strip()
    print(f"camera {cam}: {n} frame files")

# Xác minh mọi image_path từ máy local đã được ánh xạ sang dataset trên Colab.
import sys
sys.path.insert(0, REPO)
from src.pose.estimate_pose import load_frame_list, validate_frame_paths
frames = load_frame_list()
validate_frame_paths(frames)
print(f"OK: resolved {len(frames)} frame paths; first = {frames[0]['image_path']}")

In [ ]:
# Bước 6 — xác nhận torch thấy GPU (Colab co san torch/torchvision, chi can kiem tra)
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Bước 7 — chay pipeline T02 (giong het lenh chay local: python -m src.pose.estimate_pose)
%cd {REPO}
!python -m src.pose.estimate_pose

## Bước 8 — tải kết quả về máy

Sau khi script chạy xong, `outputs/poses.jsonl` phải có đúng **53631 dòng**. Cell dưới nén kết quả (`poses.jsonl`, `pose_previews/`, `reports/pose_quality.md`) lại và tải về máy qua trình duyệt.

In [ ]:
!wc -l {REPO}/outputs/poses.jsonl

RESULT_ZIP = "/content/t02_results.zip"
!cd {REPO} && zip -q -r {RESULT_ZIP} outputs/poses.jsonl outputs/pose_previews reports/pose_quality.md

from google.colab import files
files.download(RESULT_ZIP)

Sau khi tải `t02_results.zip` về, giải nén đè vào `Museum/pose_rule_anomaly/` trên máy local (`outputs/poses.jsonl`, `outputs/pose_previews/`, `reports/pose_quality.md`) rồi chạy checklist xác minh T02 theo plan `tender-jingling-bird.md`.